# Club convergence in Python: a tutorial

This notebook walks through a complete club convergence analysis with `convergence_clubs`:

1. prepare the data;
2. test whether the whole panel converges;
3. find convergence clubs and merge them;
4. check how robust the clubs are;
5. complement them with β-convergence, σ-convergence and distribution dynamics;
6. replicate a published Stata analysis;
7. export everything as a report.

It uses the GDP data of Phillips and Sul (2009), which ships with the package, and the productivity data of Mendez (2020). The test suite runs every cell, so the code always works with the installed version.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import convergence_clubs as cc

pd.set_option("display.precision", 3)

## 1. Prepare the data

The data are a pandas DataFrame with **units in rows** (the index holds the names) and **periods in columns**. Phillips and Sul work with the log of income per capita, smoothed with a Hodrick-Prescott filter to remove business cycles. They use λ = 400 for annual data.

In [ ]:
gdp = cc.load_gdp()  # levels: 152 countries x 1970-2003
trend = cc.hp_filter(np.log(gdp), lamb=400)
gdp.iloc[:5, :6]

If your data are in long format (one row per unit and year), `cc.wide_panel(df, unit=..., time=..., value=...)` reshapes them. `cc.describe_missing`, `cc.fill_gaps` and `cc.balance` help with gaps.

## 2. Does the whole panel converge?

The **log t test** regresses log(H₁/Hₜ) − 2 log(log t) on log t, where Hₜ is the cross-sectional variation, after dropping the first third of the periods. The null hypothesis is convergence, and it is **rejected when t < −1.65**.

In [ ]:
H = cc.cross_sectional_variation(trend.to_numpy())
cc.log_t_test(H, time_trim=1 / 3)

t = −159.6 is far below −1.65, so the 152 countries do not converge to a common path. The question is whether there are **groups** that do.

## 3. Convergence clubs

`find_clubs` sorts the units by their last value. It then forms a core group, adds to the core every unit that converges with it (t above c* = 0), and repeats the procedure on the units that are left.

In [ ]:
clubs = cc.find_clubs(trend)
clubs.summary()

How to read the table:
- `t_value` is each club's own log t test, and every club should have t > −1.65 (`converges`). With the default `cstar_method="fixed"` a club can fail; `find_clubs` then warns, and Phillips and Sul suggest `cstar_method="incremental"`.
- `beta` measures the speed: β ≥ 2 means convergence in levels, while 0 ≤ β < 2 means the gaps shrink only in relative terms.

The procedure tends to over-split, so adjacent clubs that converge together are merged:

In [ ]:
merged = clubs.merge("PS")
merged.summary(members=True)[["n_units", "t_value", "merged_clubs", "members"]]

In [ ]:
merged.plot(ncols=4);

Each panel shows the relative transition paths of a club's units: income relative to the cross-sectional mean, with the club average in color. The last panel compares the club averages. The clubs settle at different relative levels, which is what club convergence means.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
merged.plot.log_t("club1", ax=axes[0])
merged.plot.variation(ax=axes[1]);

Left: the log t regression of club 1, with a positive slope and t far above −1.65. Right: the cross-sectional variation H(t). Over the regression window it falls by 44–66% in the four largest clubs, while for the panel as a whole it rises by about 23%. Very small clubs (here two countries) give erratic H(t) paths, so read them with care.

## 4. How robust are the clubs?

Club memberships depend on choices such as the trimming fraction and the HAC estimator. `sensitivity` reruns the procedure over a grid of these settings. `ari_vs_baseline` is the adjusted Rand index of agreement with the first run: 1 means identical clubs, 0 means chance agreement.

In [ ]:
runs = cc.sensitivity(trend, merge=["PS"])
runs.table()

`bootstrap` perturbs the data instead of the settings. Here it clusters 100 random subsamples of 80% of the countries. A unit's **stability** is the share of runs in which it stays with its full-sample club mates.

In [ ]:
boot = cc.bootstrap(trend, 100, merge="PS", seed=0)
boot.uncertain(0.8).head(8)

In [ ]:
boot.plot_stability();

Club 1 is solid. Some members of the lower clubs move between neighboring clubs, and those memberships deserve less weight in the interpretation.

## 5. β, σ and distribution dynamics

These classical measures use the **unfiltered levels**.
- **β-convergence**: do poorer countries grow faster?
- **σ-convergence**: does the dispersion shrink over time?
- **Distribution dynamics** (Quah): how do countries move between income classes?

In [ ]:
absolute = cc.beta_convergence(gdp)
conditional = cc.beta_convergence(gdp, clubs=merged, club_effects=True)
absolute, conditional

In [ ]:
sigma = cc.sigma_convergence(gdp, clubs=merged)
sigma.trend()[["n_units", "change", "slope", "t_value", "converging"]]

In [ ]:
dynamics = cc.distribution_dynamics(gdp, lag=5, clubs=merged)
dynamics.distributions()

Together these tell a consistent story:
- no absolute β-convergence, and dispersion rises across all countries;
- yet strong convergence *within* the clubs;
- and a long-run distribution that piles up at both ends ("twin peaks").

## 6. Replicating a Stata analysis

The R package and Stata's `psecta` differ in a few rules: the core group, how c* is raised, the default trimming and rounding. `convention="Stata"` follows `psecta`. With it, the labor-productivity clubs of Mendez (2020) are reproduced exactly from the raw data:

In [ ]:
lp = cc.load_productivity("lp", sample="other")  # 82 countries, 1990-2014
stata = cc.find_clubs(cc.hp_filter(np.log(lp)), convention="Stata", time_trim=0.333)
[len(c) for c in stata], [len(c) for c in stata.merge("PS")]

## 7. Export a report

`report` runs everything above and writes a single self-contained HTML file, which can serve as a replication appendix:

In [ ]:
path = cc.report(gdp, bootstrap=50, title="Convergence clubs in GDP per capita").save("report.html")
path

## Next steps

- `merged.plot.interactive()` gives interactive charts (with the `interactive` extra), and `merged.plot.map(...)` draws maps (with the `maps` extra).
- From the command line: `convergence-clubs data.csv --log --hp-filter --merge PS --report report.html`.
- See the README for every option, and for how the results are validated against R and Stata.